# Analisis dan Prediksi Penjualan Video Game Konsol Nintendo

**Dataset:** Video Game Sales 2024 (Kaggle, oleh asaniczka; data asli dari VGChartz)
**Link:** https://www.kaggle.com/datasets/asaniczka/video-game-sales-2024
**Data yang dipakai:** subset konsol Nintendo (Wii, DS, 3DS, Wii U, Switch)
**Tujuan:** memprediksi `total_sales` (juta kopi) sebuah game. Targetnya angka kontinu, jadi ini masalah **regresi**.

**Alur kerja:**
1. Data splitting (bagi data menjadi train dan test)
2. Data loading (muat `train.csv` dan `test.csv`)
3. Data cleaning
4. EDA dan ExDA
5. Transformasi (encoding dan standardisasi)
6. Modeling dan evaluasi

> **Cara menjalankan:** upload `vgchartz-2024.csv` (dari Kaggle) ke folder Colab, lalu jalankan semua cell dari atas ke bawah.

## 0. Setup

In [ ]:
import glob
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn import linear_model
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_style("whitegrid")
pd.set_option("display.max_columns", 30)

FILE = "vgchartz-2024.csv"   # file asli dari Kaggle (upload ke folder Colab)
TARGET = "total_sales"       # kolom yang diprediksi

## 1. Data splitting
Dataset asli berisi semua konsol. Di sini kita ambil konsol Nintendo, buang duplikat, lalu bagi menjadi
**train (80%)** dan **test (20%)** dengan metode holdout (`train_test_split`).

Pembagian dilakukan **sebelum** cleaning, jadi data test tidak ikut memengaruhi keputusan cleaning.
Duplikat dibuang sebelum split karena baris kembar yang jatuh ke train dan test sekaligus membuat skor test menjadi palsu (data leakage).

In [ ]:
# Muat dataset asli. Baris yang formatnya rusak dilewati.
df_all = pd.read_csv(FILE, on_bad_lines="skip")
print("Seluruh dataset:", df_all.shape)
print(df_all["console"].value_counts().head(30))

In [ ]:
# Ambil konsol Nintendo. Sesuaikan kode dengan hasil value_counts() di atas.
KONSOL_NINTENDO = ["Wii", "DS", "3DS", "WiiU", "NS"]
df = df_all[df_all["console"].isin(KONSOL_NINTENDO)].copy()
print("Data Nintendo:", df.shape)
print(df["console"].value_counts())
assert len(df) >= 1000, "Kurang dari 1000 baris. Cek kode konsol atau tambah konsol."

In [ ]:
# Buang kolom gambar, baris duplikat, dan baris yang target-nya kosong
n0 = len(df)
df = df.drop(columns=["img"], errors="ignore").drop_duplicates()
print(f"Duplikat dibuang: {n0 - len(df)} baris")

n1 = len(df)
df = df.dropna(subset=[TARGET]).reset_index(drop=True)
print(f"Baris tanpa {TARGET} dibuang: {n1 - len(df)} baris")
print("Total data yang dibagi:", df.shape)

In [ ]:
# Bagi 80% train dan 20% test, lalu simpan sebagai train.csv dan test.csv
train_df, test_df = train_test_split(df, test_size=0.2, random_state=1)

train_df.to_csv("train.csv", index=False)
test_df.to_csv("test.csv", index=False)

print("Jumlah data       :", len(df))
print("Jumlah data latih :", len(train_df))
print("Jumlah data test  :", len(test_df))

## 2. Data loading
Muat kembali `train.csv` dan `test.csv` yang dihasilkan dari tahap splitting.

In [ ]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
print("Train:", train.shape, "| Test:", test.shape)
train.head()

## 3. Data cleaning
Langkah mengikuti materi: mengidentifikasi informasi dataset, mengatasi missing value, mengatasi outlier,
memeriksa duplikasi, dan memeriksa konsistensi format dan skala data.
Cleaning dikerjakan pada `train`, lalu aturan yang sama diterapkan ke `test`. Semua nilai yang dihitung
(median, batas outlier, daftar publisher populer) diambil dari **train saja**.

### 3.1 Mengidentifikasi informasi dataset

In [ ]:
train.info()

In [ ]:
train.describe(include="all")

In [ ]:
# Jumlah nilai yang hilang per kolom
missing_values = train.isnull().sum()
missing_values[missing_values > 0]

### 3.2 Memeriksa skala dan distribusi data

In [ ]:
# Skala data: penjualan tidak boleh negatif, skor kritikus harus berada di rentang wajar
print("Jumlah total_sales negatif :", (train[TARGET] < 0).sum())
if "critic_score" in train.columns:
    print("critic_score min / max      :", train["critic_score"].min(), "/", train["critic_score"].max())
print("Skewness total_sales        :", round(train[TARGET].skew(), 2), "(> 1 berarti miring ke kanan)")

### 3.3 Mengatasi missing value
Aturan dari materi:
- Kolom dengan missing value **>= 75%** dihapus.
- Kolom numerik lainnya diisi **median**.
- Kolom kategori diisi kategori baru **"Unknown"** (materi menyebut modus atau kategori baru).
- Penjualan regional yang kosong diisi **0** (asumsi: kosong berarti tidak ada penjualan yang tercatat di region itu).

In [ ]:
BATAS_MISSING = 75   # persen

persen = (train.isnull().mean() * 100).round(1)
print("Persentase missing value per kolom (train):")
print(persen[persen > 0].sort_values(ascending=False))

kolom_over = [c for c in persen[persen >= BATAS_MISSING].index if c != TARGET]
print(f"\nKolom dengan missing >= {BATAS_MISSING}% (akan dihapus):", kolom_over)

In [ ]:
SALES_COLS = ["na_sales", "jp_sales", "pal_sales", "other_sales"]
KOLOM_BUANG = sorted(set(kolom_over + ["last_update"]))   # last_update tidak relevan untuk analisis
print("Kolom yang dihapus:", KOLOM_BUANG)

def bersihkan(d, stat=None):
    # stat=None  : data TRAIN, statistik dihitung dari data ini
    # stat=dict  : data TEST, pakai statistik dari train
    d = d.copy()
    d = d.drop(columns=KOLOM_BUANG, errors="ignore")

    # konsistensi format teks
    for c in ["title", "console", "genre", "publisher", "developer"]:
        if c in d.columns:
            d[c] = d[c].astype("string").str.strip()
    for c in ["genre", "publisher", "developer"]:
        if c in d.columns:
            d[c] = d[c].fillna("Unknown")

    # konsistensi format tanggal -> tahun rilis
    d["release_date"] = pd.to_datetime(d["release_date"], errors="coerce")
    d["release_year"] = d["release_date"].dt.year
    d = d.drop(columns=["release_date"])

    # penanda "punya skor kritikus" dibuat sebelum nilai kosong diisi
    if "critic_score" in d.columns:
        d["has_critic_score"] = d["critic_score"].notna().astype(int)

    # penjualan regional kosong = 0
    sales_ada = [c for c in SALES_COLS if c in d.columns]
    d[sales_ada] = d[sales_ada].fillna(0)

    if stat is None:
        stat = {
            "median_year": d["release_year"].median(),
            "top_publisher": d["publisher"].value_counts().head(15).index.tolist(),
        }
        if "critic_score" in d.columns:
            stat["median_score"] = d["critic_score"].median()

    # numerik lain diisi median (dari train)
    d["release_year"] = d["release_year"].fillna(stat["median_year"])
    if "critic_score" in d.columns:
        d["critic_score"] = d["critic_score"].fillna(stat["median_score"])

    # publisher yang jarang muncul dikelompokkan jadi "Other"
    d["publisher_grp"] = np.where(d["publisher"].isin(stat["top_publisher"]), d["publisher"], "Other")
    return d, stat

train_bersih, STAT = bersihkan(train)
test_bersih, _ = bersihkan(test, stat=STAT)

print("Train:", train.shape, "->", train_bersih.shape)
print("Test :", test.shape, "->", test_bersih.shape)
sisa = train_bersih.isnull().sum()
print("\nSisa missing value (train):", sisa[sisa > 0].to_dict())

### 3.4 Memeriksa duplikasi data

In [ ]:
print("Baris duplikat (train):", train_bersih.duplicated().sum())
print("Baris duplikat (test) :", test_bersih.duplicated().sum())
train_bersih = train_bersih.drop_duplicates().reset_index(drop=True)
test_bersih = test_bersih.drop_duplicates().reset_index(drop=True)

### 3.5 Mengatasi outlier (metode IQR)
Batas bawah = Q1 - 1.5 x IQR dan batas atas = Q3 + 1.5 x IQR, dihitung dari **train**.

- **Fitur** (`release_year`, `critic_score`): outlier ditangani dengan `METODE_OUTLIER`
  (`"cap"` mengganti dengan batas terdekat, `"hapus"` membuang baris train yang outlier).
- **Target** (`total_sales`): outlier **dipertahankan**. Game best-seller seperti Mario Kart adalah data valid,
  bukan kesalahan input. Membuangnya berarti menghapus game yang justru ingin dipahami.
- Batas `critic_score` dihitung dari skor yang asli (bukan yang diisi median), karena nilai median yang banyak
  akan membuat IQR mendekati nol.

In [ ]:
NUM_FEATURES = [c for c in ["release_year", "critic_score"] if c in train_bersih.columns]
NUM_CEK = NUM_FEATURES + [TARGET]

def seri_asli(d, c):
    # untuk critic_score pakai skor asli saja (bukan hasil isian median)
    if c == "critic_score" and "has_critic_score" in d.columns:
        return d.loc[d["has_critic_score"] == 1, c]
    return d[c]

fig, axes = plt.subplots(1, len(NUM_CEK), figsize=(5 * len(NUM_CEK), 3.5))
axes = np.atleast_1d(axes)
batas, ringkas = {}, []
for ax, c in zip(axes, NUM_CEK):
    s = seri_asli(train_bersih, c)
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    bawah, atas = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    batas[c] = (bawah, atas)
    n_out = int(((s < bawah) | (s > atas)).sum())
    ringkas.append({"Kolom": c, "Q1": q1, "Q3": q3, "IQR": iqr,
                    "Batas bawah": bawah, "Batas atas": atas,
                    "Jumlah outlier": n_out, "Persen": round(n_out / len(s) * 100, 1)})
    sns.boxplot(x=s, ax=ax)
    ax.set_title(f"Box plot {c}")
plt.tight_layout(); plt.show()
pd.DataFrame(ringkas).round(3)

In [ ]:
METODE_OUTLIER = "cap"   # "cap" = ganti dengan batas terdekat | "hapus" = buang baris outlier (hanya train)

if METODE_OUTLIER == "cap":
    for c in NUM_FEATURES:
        bawah, atas = batas[c]
        train_bersih[c] = train_bersih[c].clip(bawah, atas)
        test_bersih[c] = test_bersih[c].clip(bawah, atas)
    print("Outlier fitur diganti dengan batas terdekat (cap).")
else:
    n_awal = len(train_bersih)
    mask = pd.Series(True, index=train_bersih.index)
    for c in NUM_FEATURES:
        bawah, atas = batas[c]
        mask &= train_bersih[c].between(bawah, atas)
    train_bersih = train_bersih[mask].reset_index(drop=True)
    print(f"Baris train outlier dibuang: {n_awal - len(train_bersih)} (data test tidak dibuang)")

### 3.6 Hasil cleaning

In [ ]:
ringkasan = pd.DataFrame({
    "Sebelum cleaning": [train.shape[0], train.shape[1], int(train.isnull().sum().sum())],
    "Sesudah cleaning": [train_bersih.shape[0], train_bersih.shape[1], int(train_bersih.isnull().sum().sum())],
}, index=["Jumlah baris (train)", "Jumlah kolom (train)", "Total sel kosong (train)"])
display(ringkasan)

train_bersih.to_csv("train_bersih.csv", index=False)
test_bersih.to_csv("test_bersih.csv", index=False)
train_bersih.head()

**Catatan keputusan cleaning (tulis ulang dengan kata-katamu setelah melihat hasilnya):**
- Kolom dengan missing value >= 75% dihapus karena terlalu sedikit informasinya.
- Kolom numerik yang kosong diisi median (tahan terhadap outlier), kolom kategori diisi "Unknown".
- `last_update` dihapus karena tidak relevan untuk memprediksi penjualan.
- Penjualan regional kosong diisi 0 dengan asumsi tidak ada penjualan tercatat.
- Outlier fitur ditangani dengan IQR; outlier target dipertahankan karena merupakan game best-seller yang valid.

## 4. EDA dan ExDA
EDA (exploratory) memahami pola data. ExDA (explanatory) menyampaikan temuan dengan jelas lewat visualisasi dan narasi.
Analisis memakai **data train yang sudah bersih**.

In [ ]:
# Pastikan tidak ada missing value tersisa
mv = train_bersih.isnull().sum()
persen_mv = mv / len(train_bersih) * 100
pd.DataFrame({"Missing Values": mv, "Percentage": persen_mv}).query("`Missing Values` > 0")

In [ ]:
# Histogram semua kolom numerik
num_all = train_bersih.select_dtypes(include="number").columns.tolist()
n_cols = 4
n_rows = -(-len(num_all) // n_cols)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows * 3.2))
axes = np.atleast_1d(axes).flatten()
for i, col in enumerate(num_all):
    train_bersih[col].hist(ax=axes[i], bins=20, edgecolor="black")
    axes[i].set_title(col)
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])
plt.tight_layout(); plt.show()

In [ ]:
# Distribusi kolom penting (dengan KDE)
kolom_plot = [c for c in [TARGET, "release_year", "critic_score"] if c in train_bersih.columns]
plt.figure(figsize=(5 * len(kolom_plot), 4))
for i, c in enumerate(kolom_plot, 1):
    plt.subplot(1, len(kolom_plot), i)
    sns.histplot(train_bersih[c], kde=True, bins=30)
    plt.title(f"Distribusi {c}")
plt.tight_layout(); plt.show()

In [ ]:
# Distribusi total_sales sangat miring, lihat versi log untuk perbandingan
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
sns.histplot(train_bersih[TARGET], bins=50, ax=ax[0]).set_title("total_sales (asli)")
sns.histplot(np.log1p(train_bersih[TARGET]), bins=50, ax=ax[1]).set_title("log(1 + total_sales)")
plt.tight_layout(); plt.show()

In [ ]:
# Komposisi data: konsol dan genre
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
train_bersih["console"].value_counts().plot(kind="bar", ax=ax[0], title="Jumlah game per konsol")
train_bersih["genre"].value_counts().plot(kind="bar", ax=ax[1], title="Jumlah game per genre")
plt.tight_layout(); plt.show()

In [ ]:
# Penjualan per konsol dan per genre
print(train_bersih.groupby("console")[TARGET].agg(["count", "mean", "median", "sum"]).round(3), "\n")
print(train_bersih.groupby("genre")[TARGET].agg(["count", "mean", "sum"]).sort_values("sum", ascending=False).round(3).head(10))

In [ ]:
# Tren penjualan per tahun rilis
tren = train_bersih.groupby("release_year")[TARGET].sum()
tren[tren.index > 1990].plot(kind="line", marker="o", title="Total penjualan per tahun rilis (juta kopi)")
plt.show()

**Korelasi.** Untuk menghitung korelasi, kolom kategori diubah menjadi angka dengan Label Encoding (seperti pada materi),
khusus untuk analisis korelasi ini. Untuk modeling, nanti dipakai One-Hot Encoding.

In [ ]:
df_lencoder = train_bersih.drop(columns=["title", "publisher", "developer"], errors="ignore").copy()
for col in df_lencoder.select_dtypes(exclude="number").columns:
    df_lencoder[col] = LabelEncoder().fit_transform(df_lencoder[col].astype(str))

plt.figure(figsize=(9, 7))
sns.heatmap(df_lencoder.corr(), annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Matriks korelasi")
plt.show()

In [ ]:
# Korelasi tiap kolom terhadap target
target_corr = df_lencoder.corr()[TARGET].drop(TARGET)
target_corr.abs().sort_values(ascending=False).plot(kind="bar", figsize=(10, 4),
                                                     title=f"Korelasi (absolut) dengan {TARGET}")
plt.ylabel("Koefisien korelasi"); plt.tight_layout(); plt.show()
print(target_corr.sort_values(ascending=False).round(3))

**ExDA: kesimpulan (tulis ulang dengan kata-katamu setelah melihat grafik dan angka di atas):**
- **Distribusi target:** `total_sales` sangat miring ke kanan. Sebagian besar game laku sedikit, segelintir sangat laku.
- **Konsol dan genre:** konsol dan genre apa yang rata-rata penjualannya paling tinggi? (lihat tabel groupby)
- **Tren tahun:** pada tahun rilis berapa penjualan paling besar?
- **Korelasi:** kolom `na_sales`, `jp_sales`, `pal_sales`, `other_sales` berkorelasi sangat kuat dengan `total_sales`
  karena `total_sales` adalah jumlahnya. Kolom ini **tidak dipakai sebagai fitur** karena sama dengan memberi model jawabannya.
- **Outlier:** game best-seller menjadi outlier pada `total_sales`. Outlier ini valid sehingga dipertahankan.

## 5. Transformasi data
Dilakukan **setelah** splitting, dan statistiknya dihitung dari train saja agar tidak ada kebocoran data.

- **One-Hot Encoding** untuk kolom kategori (`console`, `genre`, `publisher_grp`). Kategori game tidak punya urutan,
  dan One-Hot cocok untuk model regresi karena tidak menciptakan hubungan angka palsu antar kategori.
- **Standardisasi** (`StandardScaler`) untuk fitur numerik, supaya skalanya seragam (rata-rata 0, standar deviasi 1).
- Kolom `title`, `publisher`, `developer` tidak dipakai sebagai fitur (terlalu banyak nilai unik).
- Kolom penjualan regional tidak dipakai sebagai fitur (kebocoran target).

In [ ]:
CAT = ["console", "genre", "publisher_grp"]
BIN = [c for c in ["has_critic_score"] if c in train_bersih.columns]
fitur = CAT + NUM_FEATURES + BIN

# One-Hot Encoding, lalu samakan kolom test dengan kolom train
x_tr = pd.get_dummies(train_bersih[fitur], columns=CAT, dtype=float)
x_te = pd.get_dummies(test_bersih[fitur], columns=CAT, dtype=float)
x_te = x_te.reindex(columns=x_tr.columns, fill_value=0)

# Standardisasi: fit pada train, transform pada train dan test
scaler = StandardScaler()
x_train = x_tr.copy()
x_test = x_te.copy()
x_train[NUM_FEATURES] = scaler.fit_transform(x_tr[NUM_FEATURES])
x_test[NUM_FEATURES] = scaler.transform(x_te[NUM_FEATURES])

# Pisahkan target
y_train = train_bersih[TARGET]
y_test = test_bersih[TARGET]

print("Jumlah fitur setelah encoding:", x_train.shape[1])
print("Jumlah data latih :", len(x_train))
print("Jumlah data test  :", len(x_test))
print("\nRata-rata dan std fitur numerik (train) setelah standardisasi:")
print(x_train[NUM_FEATURES].agg(["mean", "std"]).round(2))

In [ ]:
# Perbandingan sebelum dan sesudah standardisasi
kolom = NUM_FEATURES[0]
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
sns.histplot(x_tr[kolom], kde=True, ax=ax[0]).set_title(f"{kolom}: sebelum standardisasi")
sns.histplot(x_train[kolom], kde=True, ax=ax[1]).set_title(f"{kolom}: sesudah standardisasi")
plt.tight_layout(); plt.show()

## 6. Modeling dan evaluasi
Kasus ini regresi (target kontinu). Tiga algoritma dilatih dengan konfigurasi dasar (tanpa hyperparameter tuning):

1. **Least Angle Regression (Lars)**
2. **Linear Regression**
3. **Gradient Boosting Regressor**

Sebagai pembanding ditambahkan **Baseline** (selalu menebak median `total_sales` dari train).
Metrik evaluasi: **MAE**, **MSE**, dan **R2** pada data test (RMSE ditambahkan agar satuannya sama dengan target).

In [ ]:
# Melatih model
lars = linear_model.Lars(n_nonzero_coefs=1).fit(x_train, y_train)
LR = LinearRegression().fit(x_train, y_train)
GBR = GradientBoostingRegressor(random_state=184).fit(x_train, y_train)
base = DummyRegressor(strategy="median").fit(x_train, y_train)

models = {
    "Baseline (median)": base,
    "Lars": lars,
    "Linear Regression": LR,
    "GradientBoostingRegressor": GBR,
}

In [ ]:
# Evaluasi pada data test
df_results = pd.DataFrame(columns=["MAE", "MSE", "RMSE", "R2"], dtype=float)
for nama, model in models.items():
    pred = model.predict(x_test)
    mae = mean_absolute_error(y_test, pred)
    mse = mean_squared_error(y_test, pred)
    r2 = r2_score(y_test, pred)
    df_results.loc[nama] = [mae, mse, np.sqrt(mse), r2]

df_results.round(4)

In [ ]:
# Perbandingan visual
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
df_results["MAE"].plot(kind="bar", ax=ax[0], title="MAE (lebih kecil lebih baik)")
df_results["R2"].plot(kind="bar", ax=ax[1], title="R2 (lebih besar lebih baik)")
plt.tight_layout(); plt.show()

kandidat = df_results.drop(index="Baseline (median)")
nama_terbaik = kandidat["R2"].idxmax()
print("Model terbaik berdasarkan R2:", nama_terbaik)

In [ ]:
# Prediksi vs kenyataan untuk model terbaik
model_terbaik = models[nama_terbaik]
pred_terbaik = model_terbaik.predict(x_test)

plt.figure(figsize=(5.5, 5.5))
plt.scatter(y_test, pred_terbaik, alpha=0.4)
mx = max(y_test.max(), pred_terbaik.max())
plt.plot([0, mx], [0, mx], "r--")
plt.xlabel("Penjualan sebenarnya (juta)"); plt.ylabel("Prediksi (juta)")
plt.title(f"{nama_terbaik}: prediksi vs kenyataan (test)")
plt.show()

In [ ]:
# Fitur paling berpengaruh (Gradient Boosting)
imp = pd.Series(GBR.feature_importances_, index=x_train.columns).sort_values(ascending=False)
imp.head(10).sort_values().plot(kind="barh", title="10 fitur paling berpengaruh (Gradient Boosting)")
plt.tight_layout(); plt.show()

### Menyimpan model

In [ ]:
joblib.dump(model_terbaik, "model_terbaik.joblib")

print("File hasil yang dibuat:")
for f in sorted(glob.glob("*.csv") + glob.glob("*.joblib")):
    print(" -", f)

## 7. Kesimpulan
Isi dengan hasil kamu setelah menjalankan semua cell:
1. **Data:** berapa baris awal, berapa train dan test, berapa baris setelah cleaning.
2. **Cleaning:** masalah apa yang ditemukan (missing value, duplikat, outlier) dan bagaimana ditangani.
3. **EDA dan ExDA:** konsol, genre, dan tahun apa yang paling menonjol; fitur apa yang paling berkorelasi dengan penjualan.
4. **Model:** model mana yang terbaik berdasarkan MAE, MSE, dan R2, dan apakah lebih baik dari baseline.
   Kalau R2 rendah, itu wajar karena penjualan game dipengaruhi hal yang tidak ada di data (marketing, popularitas franchise).
5. **Keterbatasan:** data hasil scraping, banyak skor kritikus yang kosong, tanpa hyperparameter tuning, dan satu kali split.